# POC-21 Phase 3 Post #3

## Analytical Track Development and Validation

**Approved Track:** Track A - Comparative Intelligence  
**Canonical Dataset:** `data/canonical/intelligence_data.csv`  
**Data Version:** 1.1.0  
**Method Version:** 1.0.0


In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CANONICAL_PATH = PROJECT_ROOT / 'data' / 'canonical' / 'intelligence_data.csv'
OUTPUT_DIR = PROJECT_ROOT / 'data-science' / 'outputs'

print('Project root:', PROJECT_ROOT)
print('Canonical dataset:', CANONICAL_PATH)
print('Dataset exists:', CANONICAL_PATH.exists())


In [ ]:
if not CANONICAL_PATH.exists():
    raise FileNotFoundError(f'Canonical dataset not found: {CANONICAL_PATH}')

df = pd.read_csv(CANONICAL_PATH)

print('Measurement rows:', len(df))
print('Columns:', list(df.columns))
print('Data versions:', sorted(df['data_version'].dropna().astype(str).unique()))
print('Distinct source incidents:', df['source_record_id'].nunique())

df.head()


## Input Validation

The analytical workflow uses only the approved canonical dataset and verifies the required fields before reconstruction.


In [ ]:
required_fields = [
    'source_record_id',
    'category',
    'entity_id',
    'shift',
    'metric_name',
    'metric_value',
    'data_version',
    'is_synthetic'
]

missing_fields = [field for field in required_fields if field not in df.columns]

if missing_fields:
    raise ValueError(f'Missing required fields: {missing_fields}')

print('Required field validation: PASS')
print('Expected data version: 1.1.0')
print('Actual data versions:', sorted(df['data_version'].dropna().astype(str).unique()))
print('All records synthetic:', bool(df['is_synthetic'].astype(bool).all()))


## Incident Reconstruction

Each source incident is reconstructed by pairing `downtime_minutes` and `failure_frequency` using `source_record_id`.


In [ ]:
working = df.copy()
working['metric_value'] = pd.to_numeric(working['metric_value'], errors='coerce')

invalid_numeric_rows = int(working['metric_value'].isna().sum())

pivot = working.pivot_table(
    index=['source_record_id', 'category', 'entity_id', 'shift'],
    columns='metric_name',
    values='metric_value',
    aggfunc='first'
).reset_index()

required_metrics = ['downtime_minutes', 'failure_frequency']

for metric in required_metrics:
    if metric not in pivot.columns:
        pivot[metric] = pd.NA

complete_incidents = pivot.dropna(subset=required_metrics).copy()

print('Invalid numeric rows:', invalid_numeric_rows)
print('Reconstructed incidents:', len(pivot))
print('Complete incidents:', len(complete_incidents))

complete_incidents.head()


## Baseline and Comparative Intelligence

The baseline is descriptive group comparison using incident count, total downtime, average downtime, total failure frequency, and average failure frequency.


In [ ]:
def calculate_group_results(data, group_field):
    result = (
        data.groupby(group_field, dropna=False)
        .agg(
            incident_count=('source_record_id', 'nunique'),
            total_downtime=('downtime_minutes', 'sum'),
            average_downtime=('downtime_minutes', 'mean'),
            total_failure_frequency=('failure_frequency', 'sum'),
            average_failure_frequency=('failure_frequency', 'mean')
        )
        .reset_index()
    )

    result[group_field] = result[group_field].fillna('(blank)').astype(str)

    result = result.sort_values(
        by=['total_downtime', 'incident_count', group_field],
        ascending=[False, False, True]
    ).reset_index(drop=True)

    result['rank_by_total_downtime'] = range(1, len(result) + 1)

    return result

category_results = calculate_group_results(complete_incidents, 'category')
machine_results = calculate_group_results(complete_incidents, 'entity_id')
shift_results = calculate_group_results(complete_incidents, 'shift')

print('Top category:')
display(category_results.head())

print('Top machine:')
display(machine_results.head())

print('Top shift:')
display(shift_results.head())


## Validation and Limitations

Validation confirms row counts, incident reconstruction, complete metric pairing, version consistency, synthetic-data status, group coverage, and deterministic ranking. Results remain descriptive because the dataset contains synthetic incidents and small observed groups.


In [ ]:
validation_summary = {
    'measurement_rows': int(len(df)),
    'distinct_source_incidents': int(df['source_record_id'].nunique()),
    'reconstructed_incidents': int(len(pivot)),
    'complete_incidents': int(len(complete_incidents)),
    'data_versions': sorted(df['data_version'].dropna().astype(str).unique().tolist()),
    'invalid_numeric_rows': invalid_numeric_rows,
    'all_records_synthetic': bool(df['is_synthetic'].astype(bool).all())
}

print(json.dumps(validation_summary, indent=2))

assert validation_summary['measurement_rows'] == 60
assert validation_summary['distinct_source_incidents'] == 30
assert validation_summary['complete_incidents'] == 30
assert validation_summary['data_versions'] == ['1.1.0']

print('\nNotebook validation checks: PASS')


## Reproducibility

The reusable scripts remain the authoritative execution path:

- `data-science/scripts/run_analytical_track.py`
- `data-science/scripts/validate_analytical_track.py`
- `data-science/scripts/export_intelligence_results.py`

This notebook provides reproducible development and validation evidence and does not introduce an alternative data source.
